In [8]:
!pip install datasets torch torchvision av


In [9]:
!pip install opendatasets opencv-python-headless torch_xla
import opendatasets as od
import os

print("Connecting to Kaggle Cloud...")

# This will prompt you to paste your Kaggle username and API key
# The data will download directly to the Colab environment
od.download("https://www.kaggle.com/datasets/bigyansubedi/cricket-bowling-action-recognition")

# Let's verify the download worked
data_dir = "cricket-bowling-action-recognition"
if os.path.exists(data_dir):
    classes = os.listdir(data_dir)
    print(f"\n✅ Success! Found the following video classes: {classes}")
else:
    print("Download failed.")

Connecting to Kaggle Cloud...
Skipping, found downloaded files in "./cricket-bowling-action-recognition" (use force=True to force download)

✅ Success! Found the following video classes: ['corrected_all_data']


In [10]:
!pip install opencv-python-headless

In [11]:
!pip install torch_xla

In [12]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class BowlingVideoDataset(Dataset):
    def __init__(self, data_dir, sequence_length=16, transform=None):
        self.data_dir = data_dir
        self.sequence_length = sequence_length
        self.transform = transform

        self.video_paths = []
        self.labels = []

        # 1. Grab all valid video files from the flat directory
        all_files = [f for f in os.listdir(data_dir) if f.endswith('.mp4') or f.endswith('.avi')]

        # 2. Extract unique class names directly from the file prefixes
        extracted_classes = set()
        for f in all_files:
            # Splits "fast_left_0000020.avi" into "fast_left"
            # It joins everything except the last part (the numbers)
            class_name = "_".join(f.split('_')[:-1])
            extracted_classes.add(class_name)

        self.classes = sorted(list(extracted_classes))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # 3. Map every video to its corresponding numerical label
        for f in all_files:
            class_name = "_".join(f.split('_')[:-1])
            self.video_paths.append(os.path.join(data_dir, f))
            self.labels.append(self.class_to_idx[class_name])

        print(f"✅ Found {len(self.video_paths)} videos across {len(self.classes)} classes:")
        print(f"   Classes detected: {self.classes}")

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        frames = self.extract_frames(video_path)

        if self.transform:
            frames = torch.stack([self.transform(frame) for frame in frames])

        return frames, label

    def extract_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames >= self.sequence_length:
            indices = np.linspace(0, total_frames - 1, self.sequence_length, dtype=int)
        else:
            indices = np.concatenate((
                np.arange(total_frames),
                np.full(max(0, self.sequence_length - total_frames), total_frames - 1)
            ))

        frames = []
        for i in range(total_frames):
            ret, frame = cap.read()
            if not ret:
                break
            if i in indices:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(frame))

        cap.release()

        while len(frames) < self.sequence_length:
            frames.append(frames[-1] if len(frames) > 0 else Image.new('RGB', (224, 224)))

        return frames[:self.sequence_length]

# --- SETUP THE LOADERS ---
video_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset_directory = "cricket-bowling-action-recognition/corrected_all_data"

master_dataset = BowlingVideoDataset(
    data_dir=dataset_directory,
    sequence_length=16,
    transform=video_transforms
)

train_size = int(0.8 * len(master_dataset))
val_size = len(master_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(master_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

print("\n🚀 Ready for TPU Training! 🚀")

✅ Found 2573 videos across 6 classes:
   Classes detected: ['fast_left', 'fast_right', 'leg_left', 'leg_right', 'off_left', 'off_right']

🚀 Ready for TPU Training! 🚀


In [13]:
import torch.optim as optim
import time
import torch
import torch.nn as nn
import torchvision.models as models

# Baseline model for Video Classification
class CricketShotBaseline(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super(CricketShotBaseline, self).__init__()
        # Load a pre-trained ResNet model
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity() 

        # Add a new fully connected layer for video classification
        self.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, C, H, W)
        batch_size, sequence_length, C, H, W = x.size()
        x = x.view(-1, C, H, W)
        features = self.resnet(x)
        features = features.view(batch_size, sequence_length, -1)
        pooled_features = torch.mean(features, dim=1) 
        output = self.fc(pooled_features)
        return output

# 1. GPU Device Allocation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training allocated to: {device}")

# 2. Model Initialization
num_classes = len(master_dataset.classes)
model = CricketShotBaseline(num_classes=num_classes).to(device)

# 3. Training Parameters
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
num_epochs = 15

print("-" * 60)
print("Starting GPU Training Phase...")
print("-" * 60)

for epoch in range(num_epochs):
    start_time = time.time()

    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_epoch_loss = running_loss / len(train_dataset)
    train_epoch_acc = (correct_train / total_train) * 100
    scheduler.step()

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_loss / len(val_dataset)
    val_epoch_acc = (correct_val / total_val) * 100
    epoch_duration = time.time() - start_time

    print(f"Epoch [{epoch+1}/{num_epochs}] | Time: {epoch_duration:.1f}s | LR: {scheduler.get_last_lr()[0]:.6f}")
    print(f"  -> Train Loss: {train_epoch_loss:.4f} | Train Acc: {train_epoch_acc:.2f}%")
    print(f"  -> Val Loss:   {val_epoch_loss:.4f} | Val Acc:   {val_epoch_acc:.2f}%")

print("-" * 60)
print("GPU Training Complete!")

Training allocated to: cuda
------------------------------------------------------------
Starting GPU Training Phase...
------------------------------------------------------------
Epoch [1/15] | Time: 508.3s | LR: 0.000905
  -> Train Loss: 1.5848 | Train Acc: 36.49%
  -> Val Loss:   2.2639 | Val Acc:   40.78%
Epoch [2/15] | Time: 505.4s | LR: 0.000655
  -> Train Loss: 1.2923 | Train Acc: 53.55%
  -> Val Loss:   1.0386 | Val Acc:   71.65%
Epoch [3/15] | Time: 510.6s | LR: 0.000345
  -> Train Loss: 1.1000 | Train Acc: 65.11%
  -> Val Loss:   1.2213 | Val Acc:   70.29%
Epoch [4/15] | Time: 505.6s | LR: 0.000095
  -> Train Loss: 0.9088 | Train Acc: 78.33%
  -> Val Loss:   0.9301 | Val Acc:   80.58%
Epoch [5/15] | Time: 503.4s | LR: 0.001000
  -> Train Loss: 0.7800 | Train Acc: 84.11%
  -> Val Loss:   0.7985 | Val Acc:   86.21%
Epoch [6/15] | Time: 506.9s | LR: 0.000976
  -> Train Loss: 0.9433 | Train Acc: 76.87%
  -> Val Loss:   0.8997 | Val Acc:   77.48%
Epoch [7/15] | Time: 506.0s | LR:

In [14]:
# Save the trained weights
torch.save(model.state_dict(), 'cricket_bowling_model_ep15.pth')